# 💄 뷰티 UGC 영상 생성 AI — **MOCK DEMO**

> **API 키 없이 전체 파이프라인 흐름을 테스트하는 데모 노트북입니다.**
> 실제 AI 결과물 대신 시각적으로 동일한 구조의 Mock 데이터를 생성합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJaeWon/Astar/blob/claude/beauty-ugc-video-generator-i10mH/beauty_ugc_video_generator_DEMO.ipynb)

## 파이프라인
| 단계 | 실제 버전 | 이 데모 |
|------|----------|---------|
| A. 기획안 생성 | Claude API | 하드코딩 샘플 JSON |
| B. 크리에이터 이미지 | DALL-E 3 | PIL 생성 Mock 이미지 |
| C. 장면 이미지 | DALL-E 3 | PIL 생성 Mock 이미지 |
| D. 영상 생성 | Google Veo3 | moviepy 생성 Mock 영상 |
| E. 영상 결합 | moviepy | moviepy (동일) |

## 0. 의존성 설치 (API 라이브러리 제외)

In [ ]:
!pip install -q moviepy Pillow requests
# moviepy 내부 의존성
!pip install -q imageio imageio-ffmpeg

## 1. 기본 설정

In [ ]:
import json
import time
import textwrap
import numpy as np
import requests
from pathlib import Path
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Markdown, Video
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import moviepy.editor as mpe
    MOVIEPY_OK = True
except Exception:
    MOVIEPY_OK = False
    print('⚠️  moviepy 없음 — Step E 스킵')

OUTPUT_DIR = Path('ugc_output_demo')
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'images').mkdir(exist_ok=True)
(OUTPUT_DIR / 'videos').mkdir(exist_ok=True)

# 파이프라인 시각화용 색상 팔레트
SCENE_COLORS = [
    '#FF6B6B', '#FFD93D', '#6BCB77', '#4D96FF', '#C77DFF', '#FF9A3C'
]

print('✅ 초기화 완료')

## 파이프라인 구조 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_facecolor('#0f0f0f')
fig.patch.set_facecolor('#0f0f0f')

steps = [
    ('INPUT\n제품이미지\n+기획초안', '#555555', 0.5),
    ('A\n기획안\n(Claude)', '#4D96FF', 2.2),
    ('B\n크리에이터\n(DALL-E3)', '#C77DFF', 3.9),
    ('C\n장면이미지\n(DALL-E3)', '#FFD93D', 5.6),
    ('D\n영상생성\n(Veo3)', '#FF6B6B', 7.3),
    ('E\n영상결합\n(moviepy)', '#6BCB77', 9.0),
]

for label, color, x in steps:
    circle = plt.Circle((x, 2.5), 0.7, color=color, zorder=3)
    ax.add_patch(circle)
    ax.text(x, 2.5, label, ha='center', va='center',
            fontsize=7.5, fontweight='bold', color='white', zorder=4,
            multialignment='center')
    if x < 9.0:
        ax.annotate('', xy=(x + 0.85, 2.5), xytext=(x + 0.7, 2.5),
                    arrowprops=dict(arrowstyle='->', color='white', lw=2), zorder=5)

ax.text(5, 4.5, '💄 뷰티 브랜드 UGC 영상 생성 AI — 전체 파이프라인',
        ha='center', va='center', fontsize=13, color='white', fontweight='bold')
ax.text(5, 0.4, '최종 출력: 30초 9:16 UGC 쇼츠 영상',
        ha='center', va='center', fontsize=10, color='#aaaaaa')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'images' / 'pipeline.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ 파이프라인 다이어그램 저장')

## 2. 입력값 — 제품 이미지 + 기획초안

In [ ]:
# 공개 이미지 URL (Unsplash 무료 이미지)
PRODUCT_IMAGE_URL = "https://images.unsplash.com/photo-1571781926291-c477ebfd024b?w=400&q=80"

USER_BRIEF = """
브랜드: 글로우랩 (GlowLab)
제품: 비타민C 브라이트닝 세럼 30ml
타겟: 20~30대 여성, 칙칙하고 피로해 보이는 피부 고민
핵심 메시지: 단 하루만에 달라지는 피부 광채
톤&무드: 밝고 생동감 있는, 진정성 있는 일상 룩
CTA: 링크 클릭 → 첫 구매 20% 할인
"""

# 제품 이미지 로드
try:
    resp = requests.get(PRODUCT_IMAGE_URL, timeout=10)
    product_img = Image.open(BytesIO(resp.content)).convert('RGB')
    print(f'✅ 제품 이미지 로드: {product_img.size}')
except Exception:
    # 인터넷 없을 경우 플레이스홀더
    product_img = Image.new('RGB', (400, 400), color='#f0e6d3')
    d = ImageDraw.Draw(product_img)
    d.text((120, 180), '제품 이미지\nPlaceholder', fill='#888888')
    print('⚠️  네트워크 없음 — 플레이스홀더 사용')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(product_img)
axes[0].set_title('📦 제품 이미지 (입력)', fontsize=11, pad=8)
axes[0].axis('off')

axes[1].axis('off')
axes[1].set_facecolor('#f8f8f8')
brief_lines = USER_BRIEF.strip().split('\n')
axes[1].text(0.05, 0.95, '📋 기획초안 (입력)',
             transform=axes[1].transAxes, fontsize=11,
             va='top', fontweight='bold')
for i, line in enumerate(brief_lines):
    axes[1].text(0.05, 0.82 - i * 0.12, line,
                 transform=axes[1].transAxes, fontsize=9,
                 va='top', color='#333333')
plt.tight_layout()
plt.show()

## STEP A — UGC 기획안 생성 (Mock: Claude API 시뮬레이션)
실제 버전: `claude-sonnet-4-6` 이 제품 이미지 분석 → 장면별 기획안 JSON 반환

In [ ]:
# ── Mock: Claude가 생성할 UGC 기획안 샘플 ─────────────────────
MOCK_SCENES = [
    {
        "scene_number": 1,
        "duration_sec": 5,
        "title": "훅 — 공감 유발",
        "description": "크리에이터가 거울 앞에서 칙칙한 피부를 보며 한숨을 쉬는 장면. 자연스러운 표정 연기.",
        "voiceover": "아침마다 피부가 왜 이렇게 칙칙하지...😩",
        "visual_style": "핸드헬드, 클로즈업, 따뜻한 자연광, 아침 분위기",
        "product_focus": "아직 제품 미노출 — 공감대 형성 구간",
        "first_scene_prompt": "Korean woman in her 20s looking at bathroom mirror with tired expression, morning light, no makeup, authentic selfie style, vertical 9:16",
        "last_scene_prompt": "Same Korean woman sighing while touching her dull skin in mirror reflection, soft morning light, realistic UGC photo"
    },
    {
        "scene_number": 2,
        "duration_sec": 5,
        "title": "제품 발견",
        "description": "책상 위에 세럼 박스가 택배로 도착. 크리에이터가 설레는 표정으로 언박싱.",
        "voiceover": "드디어 왔다!! 유명하다던 그 세럼 ✨",
        "visual_style": "탑뷰 + 핸드헬드 혼합, 밝고 화사한 조명",
        "product_focus": "제품 박스 언박싱 클로즈업, 브랜드 로고 노출",
        "first_scene_prompt": "Delivery box on white desk, Korean woman's hands opening it excitedly, bright natural light, product unboxing, vertical 9:16 UGC style",
        "last_scene_prompt": "Vitamin C serum bottle held up by Korean woman with excited expression, clean white background, product hero shot, vertical 9:16"
    },
    {
        "scene_number": 3,
        "duration_sec": 6,
        "title": "사용 장면",
        "description": "세럼을 손등에 펴발라 질감을 보여주고, 얼굴에 가볍게 두드려 흡수시키는 장면.",
        "voiceover": "와 이 촉촉함... 물광 느낌 장난 아닌데?",
        "visual_style": "매크로 클로즈업, 슬로우모션 질감 강조",
        "product_focus": "세럼 텍스처 클로즈업, 피부 흡수 장면",
        "first_scene_prompt": "Close-up of serum drops on Korean woman's hand, golden vitamin C liquid texture, macro shot, warm light, vertical 9:16",
        "last_scene_prompt": "Korean woman gently patting serum into glowing skin, close-up face, dewy skin effect, soft bokeh background, vertical 9:16"
    },
    {
        "scene_number": 4,
        "duration_sec": 6,
        "title": "비포-애프터 비교",
        "description": "화면을 반으로 나눠 사용 전/후 피부톤 변화를 보여주는 스플릿 장면.",
        "voiceover": "하루만에 이렇게 달라졌어요 (진짜임)",
        "visual_style": "스플릿 화면, 밝은 스튜디오 조명, 비교 강조",
        "product_focus": "피부 변화가 주인공, 제품은 모서리에 작게 노출",
        "first_scene_prompt": "Split screen comparison, left side dull tired Korean woman skin before, right side glowing bright skin after, clean white background, vertical 9:16",
        "last_scene_prompt": "Korean woman with radiant glowing skin smiling confidently, bright even skin tone, natural makeup, dewy finish, vertical 9:16 portrait"
    },
    {
        "scene_number": 5,
        "duration_sec": 5,
        "title": "일상 속 자연스러운 사용",
        "description": "밝아진 피부로 외출 준비를 마치고 카메라 보며 윙크하는 장면.",
        "voiceover": "이제 파데 없이도 자신있어 💛",
        "visual_style": "밝은 자연광, 생동감 있는 컬러그레이딩",
        "product_focus": "배경에 세럼 자연스럽게 배치",
        "first_scene_prompt": "Korean woman in casual outfit getting ready to go out, bright morning light, glowing skin visible, serum bottle in background, vertical 9:16",
        "last_scene_prompt": "Happy Korean woman winking at camera with bright glowing skin, natural confident expression, lifestyle UGC photo, vertical 9:16"
    },
    {
        "scene_number": 6,
        "duration_sec": 5,
        "title": "CTA — 구매 유도",
        "description": "제품을 들고 카메라에 가까이 보여주며 할인 정보를 알려주는 클로징 장면.",
        "voiceover": "첫 구매 20% 할인 중이래요 — 링크는 프로필에 🔗",
        "visual_style": "밝고 깔끔한 배경, 제품 메인, 텍스트 오버레이",
        "product_focus": "제품을 카메라 정면으로 클로즈업, 가격/할인 텍스트 강조",
        "first_scene_prompt": "Korean woman holding vitamin C serum bottle close to camera, bright clean background, product showcase, call-to-action style, vertical 9:16",
        "last_scene_prompt": "Product flat lay with serum bottle, soft pink background, '20% OFF' text overlay, beauty aesthetic, vertical 9:16 UGC style"
    }
]

scenes = MOCK_SCENES
with open(OUTPUT_DIR / 'ugc_plan.json', 'w', encoding='utf-8') as f:
    json.dump(scenes, f, ensure_ascii=False, indent=2)

total_sec = sum(s['duration_sec'] for s in scenes)
print(f'✅ STEP A (Mock) 완료: {len(scenes)}개 장면, 총 {total_sec}초\n')

# ── 기획안 시각화 타임라인 ────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 2.5))
ax.set_facecolor('#fafafa')
ax.set_xlim(0, total_sec)
ax.set_ylim(0, 2)
ax.axis('off')
ax.set_title(f'📋 STEP A — UGC 기획안 타임라인 (총 {total_sec}초)', fontsize=12, pad=10)

pos = 0
for i, s in enumerate(scenes):
    dur = s['duration_sec']
    color = SCENE_COLORS[i % len(SCENE_COLORS)]
    rect = mpatches.FancyBboxPatch(
        (pos + 0.1, 0.4), dur - 0.2, 1.1,
        boxstyle='round,pad=0.05', fc=color, ec='white', lw=2, alpha=0.9
    )
    ax.add_patch(rect)
    label = f"Scene {s['scene_number']}\n{s['title']}\n({dur}초)"
    ax.text(pos + dur / 2, 0.95, label, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')
    pos += dur

# 타임라인 눈금
for t in range(0, total_sec + 1, 5):
    ax.axvline(x=t, color='#cccccc', lw=0.8, ls='--')
    ax.text(t, 0.15, f'{t}s', ha='center', fontsize=7.5, color='#888888')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'images' / 'timeline.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n[기획안 상세]')
for s in scenes:
    print(f"  [{s['scene_number']}] {s['title']:15s} | {s['duration_sec']}초 | {s['voiceover']}")

## STEP B — 가상 크리에이터 이미지 (Mock: DALL-E 3 시뮬레이션)
실제 버전: DALL-E 3 `1024×1792` HD Natural 스타일로 크리에이터 페르소나 생성

In [ ]:
def make_mock_creator_image(size=(540, 960)) -> Image.Image:
    """DALL-E 3 결과 대신 시각적으로 표현한 Mock 크리에이터 이미지"""
    img = Image.new('RGB', size, color='#fde8d8')
    d = ImageDraw.Draw(img)

    # 배경 그라디언트 효과 (간단한 색상 블록)
    for y in range(size[1]):
        r = int(253 - y * 0.05)
        g = int(232 - y * 0.08)
        b = int(216 - y * 0.03)
        d.line([(0, y), (size[0], y)], fill=(max(r,180), max(g,160), max(b,180)))

    # 인물 실루엣 (단순 도형)
    cx, cy = size[0]//2, size[1]//3
    # 얼굴
    d.ellipse([cx-90, cy-100, cx+90, cy+80], fill='#f5d5b5', outline='#e8b090', width=3)
    # 머리카락
    d.ellipse([cx-95, cy-115, cx+95, cy-40], fill='#3d2b1f')
    d.rectangle([cx-95, cy-80, cx-70, cy+60], fill='#3d2b1f')
    d.rectangle([cx+70, cy-80, cx+95, cy+60], fill='#3d2b1f')
    # 눈
    d.ellipse([cx-45, cy-35, cx-20, cy-18], fill='#3d2b1f')
    d.ellipse([cx+20, cy-35, cx+45, cy-18], fill='#3d2b1f')
    # 입
    d.arc([cx-25, cy+20, cx+25, cy+50], 0, 180, fill='#c97060', width=4)
    # 몸통 (흰 스웨터)
    d.ellipse([cx-120, cy+65, cx+120, cy+320], fill='#f0ede8', outline='#ddd', width=2)

    # 레이블
    d.rectangle([0, size[1]-140, size[0], size[1]], fill=(0,0,0,180))
    d.text((size[0]//2-120, size[1]-120), '[B] DALL-E 3 생성 예정', fill='white')
    d.text((size[0]//2-140, size[1]-90), '가상 크리에이터 페르소나', fill='#FFD93D')
    d.text((size[0]//2-150, size[1]-60), '한국인 여성, 20대 후반, 데이룩', fill='#aaaaaa')
    d.text((size[0]//2-100, size[1]-30), '9:16  |  1024×1792', fill='#888888')

    return img

print('⏳ STEP B (Mock): 가상 크리에이터 이미지 생성 중...')
creator_img = make_mock_creator_image()
creator_path = OUTPUT_DIR / 'images' / 'creator_model.png'
creator_img.save(creator_path)

fig, ax = plt.subplots(figsize=(4, 7))
ax.imshow(creator_img)
ax.set_title('STEP B — 가상 크리에이터 이미지 (Mock)', fontsize=10, pad=6)
ax.axis('off')
plt.tight_layout()
plt.show()
print(f'✅ STEP B (Mock) 완료: {creator_path}')

## STEP C — 장면별 1st / Last Scene 이미지 (Mock)
실제 버전: 기획안 프롬프트 + 크리에이터 스타일로 각 장면 시작·끝 프레임을 DALL-E 3 생성

In [ ]:
def make_mock_scene_image(
    scene: dict, frame_type: str, color: str, size=(540, 960)
) -> Image.Image:
    """장면 이미지 Mock — 색상 배경 + 장면 정보 텍스트"""
    # 배경
    img = Image.new('RGB', size, color=color)
    d = ImageDraw.Draw(img, 'RGBA')

    # 그라디언트 오버레이
    for y in range(size[1]):
        alpha = int(255 * (y / size[1]) * 0.45)
        d.line([(0, y), (size[0], y)], fill=(0, 0, 0, alpha))

    # 인물 실루엣 (간단)
    cx, cy = size[0]//2, size[1]//3
    d.ellipse([cx-70, cy-90, cx+70, cy+60], fill=(245, 213, 181, 200))
    d.ellipse([cx-75, cy-105, cx+75, cy-30], fill=(61, 43, 31, 220))
    d.ellipse([cx-100, cy+55, cx+100, cy+280], fill=(240, 237, 232, 200))

    # 장면 정보 오버레이 (하단)
    d.rectangle([0, size[1]-220, size[0], size[1]], fill=(0, 0, 0, 180))

    frame_label = '1st Frame ▶' if frame_type == 'first' else '◀ Last Frame'
    title_color = '#FFD93D' if frame_type == 'first' else '#FF6B6B'

    lines = [
        (f'Scene {scene["scene_number"]}  |  {frame_label}', title_color, 14, size[1]-210),
        (f'📌 {scene["title"]}', 'white', 13, size[1]-178),
        (f'⏱ {scene["duration_sec"]}초', '#aaaaaa', 11, size[1]-150),
        (f'🎬 {scene["visual_style"][:30]}...', '#cccccc', 10, size[1]-125),
        (f'💬 {scene["voiceover"][:28]}...', '#eeeeee', 10, size[1]-100),
        ('DALL-E 3 생성 예정', '#666666', 9, size[1]-55),
        ('9:16  |  1024×1792  |  HD Natural', '#555555', 8, size[1]-35),
    ]
    for text, fill, _, y in lines:
        d.text((20, y), text, fill=fill)

    return img


print('⏳ STEP C (Mock): 장면별 시작/끝 프레임 이미지 생성 중...')
scene_images = []

for i, scene in enumerate(scenes):
    color = SCENE_COLORS[i % len(SCENE_COLORS)]
    n = scene['scene_number']

    first_img = make_mock_scene_image(scene, 'first', color)
    first_path = str(OUTPUT_DIR / 'images' / f'scene_{n:02d}_first.png')
    first_img.save(first_path)

    last_img = make_mock_scene_image(scene, 'last', color)
    last_path = str(OUTPUT_DIR / 'images' / f'scene_{n:02d}_last.png')
    last_img.save(last_path)

    scene_images.append({
        'scene': scene, 'color': color,
        'first_path': first_path, 'last_path': last_path,
        'first_img': first_img, 'last_img': last_img
    })
    print(f'  ✓ Scene {n}: {scene["title"]} — 1st + Last frame 저장')

print(f'\n✅ STEP C (Mock) 완료: {len(scene_images)*2}장 이미지 저장')

# ── 장면 이미지 그리드 시각화 ─────────────────────────────────
n_scenes = len(scene_images)
fig, axes = plt.subplots(2, n_scenes, figsize=(n_scenes * 2.2, 9))
fig.suptitle('STEP C — 장면별 1st / Last Frame 이미지 (Mock)', fontsize=12, y=1.01)

for col, si in enumerate(scene_images):
    for row, (key, label) in enumerate([('first_img', '1st Frame'), ('last_img', 'Last Frame')]):
        axes[row][col].imshow(si[key].resize((270, 480)))
        axes[row][col].set_title(
            f'S{si["scene"]["scene_number"]} {label}', fontsize=8, pad=4
        )
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'images' / 'scene_frames_preview.png', dpi=100, bbox_inches='tight')
plt.show()

## STEP D — 장면별 영상 생성 (Mock: Veo3 시뮬레이션)
실제 버전: Veo3 `veo-3.0-generate-preview` 모델에 1st/Last frame + 프롬프트 입력 → 장면 영상

In [ ]:
def make_mock_scene_video(
    scene: dict,
    first_img: Image.Image,
    last_img: Image.Image,
    output_path: str,
    fps: int = 24
) -> str:
    """Veo3 시뮬레이션 — 1st→Last 이미지 페이드 전환 MP4 생성"""
    duration = scene['duration_sec']
    n_frames = int(duration * fps)
    w, h = 270, 480  # 데모용 소형 해상도 (9:16)

    first_arr = np.array(first_img.resize((w, h))).astype(np.float32)
    last_arr  = np.array(last_img.resize((w, h))).astype(np.float32)

    def make_frame(t):
        """시간 t에서의 프레임: 1st→last 선형 보간 + 텍스트 오버레이"""
        alpha = t / duration
        frame = ((1 - alpha) * first_arr + alpha * last_arr).astype(np.uint8)
        pil_frame = Image.fromarray(frame)
        d = ImageDraw.Draw(pil_frame)

        # 자막 오버레이
        d.rectangle([0, h-45, w, h], fill=(0, 0, 0, 160))
        d.text((8, h-40), f"{scene['voiceover'][:20]}...", fill='white')
        d.text((8, h-22), f"Scene {scene['scene_number']} | {t:.1f}s / {duration}s",
               fill='#FFD93D')
        return np.array(pil_frame)

    clip = mpe.VideoClip(make_frame, duration=duration)
    clip = clip.set_fps(fps)
    clip.write_videofile(
        output_path, fps=fps, codec='libx264',
        audio=False, verbose=False, logger=None
    )
    clip.close()
    return output_path


video_paths = []

if not MOVIEPY_OK:
    print('⚠️  moviepy 없음 — Step D 스킵')
else:
    print('⏳ STEP D (Mock): Veo3 시뮬레이션 영상 생성 중...')
    for si in scene_images:
        scene = si['scene']
        n = scene['scene_number']
        out = str(OUTPUT_DIR / 'videos' / f'scene_{n:02d}.mp4')
        print(f'  [Scene {n}] {scene["title"]} ({scene["duration_sec"]}초) 생성 중...')
        path = make_mock_scene_video(
            scene, si['first_img'], si['last_img'], out
        )
        video_paths.append(path)
        print(f'    ✅ 저장: {path}')

    print(f'\n✅ STEP D (Mock) 완료: {len(video_paths)}개 장면 영상 생성')

    # 첫 번째 장면 영상 미리보기
    if video_paths:
        print('\n[Scene 1 미리보기]')
        display(Video(video_paths[0], width=200))

## STEP E — 장면 영상 결합 → 최종 30초 UGC 영상

In [ ]:
FINAL_VIDEO_PATH = str(OUTPUT_DIR / 'final_ugc_video_demo.mp4')

if not MOVIEPY_OK or not video_paths:
    print('⚠️  영상 결합 불가 — moviepy 없거나 생성된 영상이 없습니다.')
else:
    print('⏳ STEP E: 장면 영상 결합 중...')
    clips = []
    for i, vp in enumerate(video_paths):
        c = mpe.VideoFileClip(vp)
        if i > 0:
            c = c.fadein(0.3)
        if i < len(video_paths) - 1:
            c = c.fadeout(0.3)
        clips.append(c)
        print(f'  ✓ Scene {i+1} 로드: {c.duration:.1f}초')

    final = mpe.concatenate_videoclips(clips, method='compose')
    print(f'\n  결합 총 길이: {final.duration:.1f}초')
    final.write_videofile(
        FINAL_VIDEO_PATH, fps=24, codec='libx264',
        audio=False, verbose=False, logger=None
    )
    for c in clips: c.close()
    final.close()

    print(f'\n✅ STEP E 완료! 최종 영상: {FINAL_VIDEO_PATH}')
    display(Video(FINAL_VIDEO_PATH, width=280))

## 최종 결과 대시보드

In [ ]:
fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor('#0f0f0f')
fig.suptitle('💄 뷰티 UGC 영상 생성 AI — 결과 대시보드 (Mock Demo)',
             color='white', fontsize=14, y=0.98)

# 상단: 타임라인
ax_timeline = fig.add_axes([0.02, 0.78, 0.96, 0.14])
ax_timeline.set_facecolor('#1a1a1a')
ax_timeline.set_xlim(0, total_sec)
ax_timeline.set_ylim(0, 1)
ax_timeline.axis('off')
ax_timeline.set_title('UGC 콘텐츠 타임라인', color='#aaaaaa', fontsize=10, loc='left', pad=4)

pos = 0
for i, s in enumerate(scenes):
    dur = s['duration_sec']
    color = SCENE_COLORS[i % len(SCENE_COLORS)]
    rect = mpatches.FancyBboxPatch(
        (pos + 0.05, 0.15), dur - 0.1, 0.7,
        boxstyle='round,pad=0.02', fc=color, ec='#0f0f0f', lw=1.5, alpha=0.9
    )
    ax_timeline.add_patch(rect)
    ax_timeline.text(pos + dur / 2, 0.52, f"S{s['scene_number']}\n{s['duration_sec']}초",
                     ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    pos += dur

# 중단: 장면 이미지 그리드 (1st frame만)
n_cols = len(scene_images)
for col, si in enumerate(scene_images):
    left = 0.02 + col * (0.96 / n_cols)
    ax = fig.add_axes([left + 0.005, 0.36, 0.96/n_cols - 0.01, 0.36])
    ax.imshow(si['first_img'].resize((200, 356)))
    ax.set_title(
        f"Scene {si['scene']['scene_number']}",
        color=si['color'], fontsize=8, pad=3
    )
    ax.axis('off')

# 하단: 요약 정보
ax_info = fig.add_axes([0.02, 0.02, 0.96, 0.30])
ax_info.set_facecolor('#1a1a1a')
ax_info.axis('off')

summary_lines = [
    ('📊 파이프라인 요약', 'white', 14, 0.88),
    (f'  총 장면 수: {len(scenes)}개  |  총 길이: {total_sec}초  |  포맷: 9:16 세로형', '#cccccc', 10, 0.72),
    ('  STEP A  Claude claude-sonnet-4-6  →  UGC 기획안 JSON (장면별 프롬프트 포함)', '#4D96FF', 9, 0.58),
    ('  STEP B  DALL-E 3  →  가상 크리에이터 페르소나 이미지 (1024×1792, HD Natural)', '#C77DFF', 9, 0.46),
    ('  STEP C  DALL-E 3 × (장면수 × 2)  →  각 장면 1st/Last frame 이미지', '#FFD93D', 9, 0.34),
    ('  STEP D  Google Veo3 veo-3.0-generate-preview  →  장면별 4~8초 MP4', '#FF6B6B', 9, 0.22),
    ('  STEP E  moviepy concatenate  →  크로스페이드 결합  →  최종 30초 MP4', '#6BCB77', 9, 0.10),
]
for text, color, size, y in summary_lines:
    ax_info.text(0.01, y, text, transform=ax_info.transAxes,
                 color=color, fontsize=size, va='center',
                 fontweight='bold' if size >= 12 else 'normal')

plt.savefig(OUTPUT_DIR / 'images' / 'result_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

# 출력 파일 목록
print('\n📁 생성된 파일 목록:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size // 1024
        print(f'  {str(p):50s}  ({size_kb} KB)')